# Stock Trend Forecasting with LSTM
## Practical Task 2: RNN/LSTM for Stock Trend Forecasting (10 Marks)

**Objective:** Build an LSTM model to predict the next day's closing price direction (up/down).

## 1. Import Libraries

In [ ]:
!pip install yfinance -q

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)
print(f'TensorFlow: {tf.__version__}')

## 2. Download Stock Data

In [ ]:
data = yf.download('AAPL', start='2019-01-01', end='2024-12-31')
print(f'Shape: {data.shape}')
data.head()

## 3. Visualize Closing Prices

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(data.index, data['Close'], label='Close Price')
plt.title('AAPL Stock Closing Prices')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True)
plt.show()

## 4. Preprocess Data: Normalize and Create Sequences

In [ ]:
# Extract and normalize closing prices
close_prices = data['Close'].values.reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_prices = scaler.fit_transform(close_prices)

print(f'Original range: ${close_prices.min():.2f} - ${close_prices.max():.2f}')
print(f'Scaled range: {scaled_prices.min():.4f} - {scaled_prices.max():.4f}')

In [ ]:
# Create sequences and binary labels
def create_sequences(data, seq_length=15):
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i-seq_length:i, 0])
        # Binary: 1 if next day up, 0 if down
        y.append(1 if data[i, 0] > data[i-1, 0] else 0)
    return np.array(X), np.array(y)

SEQUENCE_LENGTH = 15
X, y = create_sequences(scaled_prices, SEQUENCE_LENGTH)

print(f'Sequences: {len(X)}')
print(f'X shape: {X.shape}')
print(f'Up days: {np.sum(y==1)} ({np.sum(y==1)/len(y)*100:.1f}%)')
print(f'Down days: {np.sum(y==0)} ({np.sum(y==0)/len(y)*100:.1f}%)')

## 5. Split Data (70/15/15)

In [ ]:
train_size = int(len(X) * 0.70)
val_size = int(len(X) * 0.15)

X_train, y_train = X[:train_size], y[:train_size]
X_val, y_val = X[train_size:train_size+val_size], y[train_size:train_size+val_size]
X_test, y_test = X[train_size+val_size:], y[train_size+val_size:]

# Reshape for LSTM
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print(f'Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)')
print(f'Val: {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)')
print(f'Test: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)')

## 6. Build LSTM Model

In [ ]:
model = models.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, 1)),
    layers.Dropout(0.3),
    layers.LSTM(32, return_sequences=False),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 7. Train the Model

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

## 8. Evaluate on Test Set

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

test_accuracy = accuracy_score(y_test, y_pred)
print(f'\nTest Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

## 9. Baseline Comparison

In [ ]:
baseline_up = accuracy_score(y_test, np.ones_like(y_test))
baseline_down = accuracy_score(y_test, np.zeros_like(y_test))
baseline_random = accuracy_score(y_test, np.random.randint(0, 2, len(y_test)))

print(f'Always UP:   {baseline_up:.4f} ({baseline_up*100:.2f}%)')
print(f'Always DOWN: {baseline_down:.4f} ({baseline_down*100:.2f}%)')
print(f'Random:      {baseline_random:.4f} ({baseline_random*100:.2f}%)')
print(f'LSTM Model:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

improvement = (test_accuracy - max(baseline_up, baseline_down)) * 100
print(f'\nImprovement over best baseline: {improvement:.2f}%')

## 10. Plot Training History

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Down', 'Up'], yticklabels=['Down', 'Up'])
plt.title('Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.show()

print(cm)

## 12. Classification Report

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Down', 'Up']))

## 13. Visualize Predictions

In [ ]:
n = 100
plt.figure(figsize=(14, 6))
plt.plot(range(n), y_test[:n], 'go-', label='Actual', alpha=0.7, markersize=4)
plt.plot(range(n), y_pred[:n], 'rx-', label='Predicted', alpha=0.7, markersize=4)
plt.title(f'Actual vs Predicted (First {n} samples)')
plt.xlabel('Sample')
plt.ylabel('Direction (0=Down, 1=Up)')
plt.legend()
plt.grid(True)
plt.show()

## 14. Analysis and Discussion

### Sequence Length Choice (15 days)
- Captures roughly 3 weeks of trading activity
- Balances short-term trend detection against noise
- Too short: misses meaningful patterns
- Too long: introduces irrelevant older data

### Preprocessing Choices
- **MinMax Scaling**: Normalizes prices to [0, 1] so the LSTM doesn't get biased by absolute price values
- **Binary Labels**: Simplifies the problem to direction prediction (up vs down)
- **70/15/15 Split**: Keeps temporal order intact — no shuffling — to avoid data leakage

### Model Architecture
- **Two LSTM layers (64 → 32 units)**: First layer captures low-level temporal patterns, second layer learns higher-level dependencies
- **Dropout (0.3, 0.3, 0.2)**: Applied after each layer to reduce overfitting
- **Sigmoid output**: Appropriate for binary classification

### Is the LSTM Better Than Baseline?
Check the improvement percentage printed in Section 9. A margin above 5% suggests the model is picking up real patterns. Below that, the improvement is marginal and could be attributed to chance.

### Why Stock Prediction is Challenging
1. **Market Efficiency**: Prices already reflect all publicly available information
2. **Random Walk Theory**: Daily price movements are largely unpredictable
3. **External Factors**: News, earnings, macroeconomics — none of which are in price data alone
4. **Non-Stationarity**: Market behavior shifts over time; past patterns don't always repeat
5. **High Noise**: Daily prices carry a lot of noise relative to signal
6. **Limited Features**: Using only closing prices ignores volume, sentiment, and other indicators

### Potential Improvements
- Add technical indicators: Volume, moving averages, RSI, MACD
- Incorporate sentiment analysis from financial news
- Experiment with different sequence lengths
- Try attention mechanisms or Transformer-based architectures
- Use ensemble methods combining multiple models

### ⚠️ Disclaimer
This model is for **educational purposes only**. Do NOT use it for actual trading decisions.

## 15. Save Model (Optional)

In [ ]:
model.save('stock_lstm_model.h5')
print('Model saved!')